In [6]:
# ============================================================
# ATIVIDADE COMPLETA - PROCESSANDO DOCUMENTOS DO ZIP
# ============================================================

import os
import sys
import json
import re
import zipfile
import shutil
from pathlib import Path
from typing import List, Dict, Any
from datetime import datetime
import time

# ============================================================
# 1. INSTALAÇÃO DE DEPENDÊNCIAS
# ============================================================
!pip install -q langchain-core langchain-text-splitters sentence-transformers langchain-huggingface

print(" Dependências instaladas!")

# ============================================================
# 2. IMPORTAÇÕES
# ============================================================
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter
)
from langchain_huggingface import HuggingFaceEmbeddings

print(" Bibliotecas importadas!")

# ============================================================
# 3. DESCOMPACTAR O ARQUIVO ZIP
# ============================================================

ZIP_PATH = Path("/content/drive-download-20260812T171922Z-1-001.zip")
EXTRACT_DIR = Path("/content/documentos_extraidos")

print("\n" + "="*60)
print(" DESCOMPACTANDO ARQUIVOS")
print("="*60)

# Remove diretório se existir
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Descompacta
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print(f" Arquivos descompactados em: {EXTRACT_DIR}")
except Exception as e:
    print(f" Erro ao descompactar: {e}")
    print("Tentando método alternativo...")
    import subprocess
    subprocess.check_call(["unzip", "-q", str(ZIP_PATH), "-d", str(EXTRACT_DIR)])
    print(" Descompactado com unzip!")

# ============================================================
# 4. ENCONTRAR ARQUIVOS MARKDOWN
# ============================================================

print("\n" + "="*60)
print(" PROCURANDO ARQUIVOS MARKDOWN")
print("="*60)

# Procura por arquivos .md em toda a estrutura
md_files = list(EXTRACT_DIR.rglob("*.md"))

# Se não encontrar .md, procura .txt ou outros
if not md_files:
    md_files = list(EXTRACT_DIR.rglob("*.txt"))

# Se ainda não encontrar, procura qualquer arquivo que possa ser lido como texto
if not md_files:
    md_files = list(EXTRACT_DIR.rglob("*"))
    # Filtra apenas arquivos (não diretórios)
    md_files = [f for f in md_files if f.is_file() and not f.suffix in ['.zip', '.pdf']]

    # Pega apenas os primeiros 12 arquivos para processar
    md_files = md_files[:12]

print(f" Encontrados {len(md_files)} arquivos para processar")

for i, f in enumerate(md_files[:5], 1):  # Mostra os 5 primeiros
    print(f"   {i}. {f.name} (em {f.parent.name})")

if len(md_files) > 5:
    print(f"   ... e mais {len(md_files)-5} arquivos")

# ============================================================
# 5. MODELO DE EMBEDDINGS
# ============================================================

print("\n" + "="*60)
print(" CARREGANDO MODELO DE EMBEDDINGS")
print("="*60)

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# Testa o modelo
test_embedding = embeddings.embed_query("teste")
EMBEDDING_DIMENSION = len(test_embedding)

print(f" Modelo carregado: {EMBEDDING_MODEL}")
print(f" Dimensão: {EMBEDDING_DIMENSION}")

# ============================================================
# 6. EXERCÍCIO 1 - CRIANDO DOCUMENTS NA MÃO
# ============================================================

print("\n" + "="*60)
print("EXERCÍCIO 1 - CRIANDO DOCUMENTS NA MÃO")
print("="*60)

documentos_manuais = [
    Document(
        page_content="Embeddings são representações vetoriais densas de texto que capturam significado semântico.",
        metadata={
            "fonte": "arquivo_01.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "embeddings",
            "autor": "Aluno",
            "dificuldade": "intermediario",
            "tags": ["embedding", "vetor", "semantica"]
        }
    ),
    Document(
        page_content="O chunking é o processo de dividir documentos grandes em pedaços menores e gerenciáveis.",
        metadata={
            "fonte": "arquivo_01.md",
            "pagina": 2,
            "tipo": "pratica",
            "tema": "chunking",
            "autor": "Aluno",
            "dificuldade": "iniciante",
            "tags": ["chunking", "divisao", "documentos"]
        }
    ),
    Document(
        page_content="RAG (Retrieval-Augmented Generation) combina busca semântica com geração de texto por LLMs.",
        metadata={
            "fonte": "arquivo_02.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "rag",
            "autor": "Aluno",
            "dificuldade": "avancado",
            "tags": ["rag", "recuperacao", "geracao"]
        }
    ),
    Document(
        page_content="Tokenização é o processo de converter texto em tokens, que são unidades básicas para modelos de linguagem.",
        metadata={
            "fonte": "arquivo_02.md",
            "pagina": 2,
            "tipo": "pratica",
            "tema": "tokenizacao",
            "autor": "Aluno",
            "dificuldade": "intermediario",
            "tags": ["tokenizacao", "tokens", "nlp"]
        }
    ),
    Document(
        page_content="A similaridade por cosseno é uma métrica comum para comparar embeddings e encontrar documentos semelhantes.",
        metadata={
            "fonte": "arquivo_03.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "embeddings",
            "autor": "Aluno",
            "dificuldade": "intermediario",
            "tags": ["similaridade", "cosseno", "metrica"]
        }
    ),
    Document(
        page_content="Vector stores são bancos de dados otimizados para armazenar e buscar embeddings de forma eficiente.",
        metadata={
            "fonte": "arquivo_03.md",
            "pagina": 2,
            "tipo": "pratica",
            "tema": "rag",
            "autor": "Aluno",
            "dificuldade": "avancado",
            "tags": ["vectorstore", "busca", "indexacao"]
        }
    )
]

print(f"\n Documentos manuais criados: {len(documentos_manuais)}")

# Exibe os documentos
print("\n LISTA DE DOCUMENTOS:")
for i, doc in enumerate(documentos_manuais, 1):
    print(f"\nDocumento {i}:")
    print(f"  Conteúdo: {doc.page_content[:80]}...")
    print(f"  Metadados: {doc.metadata}")

# ============================================================
# 7. RESPOSTAS EXERCÍCIO 1
# ============================================================

print("\n" + "="*60)
print("RESPOSTAS - EXERCÍCIO 1")
print("="*60)

print("\n Que tipos de dado são aceitos dentro de metadata?")
print(" Resposta: O metadata aceita qualquer tipo serializável em JSON:")
print("   - Strings: 'fonte': 'arquivo.md'")
print("   - Números: 'pagina': 1")
print("   - Booleanos: 'relevante': True")
print("   - Listas: 'tags': ['embedding', 'vetor']")
print("   - Dicionários aninhados: 'info': {'versao': '1.0'}")
print("   - Observação: Valores não serializáveis (ex: objetos Python) causarão erro.")

print("\n O que acontece se você criar um Document sem passar metadata?")
print("✅ Resposta: O metadata é automaticamente definido como um dicionário vazio {}.")
print("   Exemplo: Document(page_content='texto') -> metadata = {}")

# ============================================================
# 8. EXERCÍCIO 2 - SCHEMA DE METADADOS
# ============================================================

print("\n" + "="*60)
print("EXERCÍCIO 2 - SCHEMA DE METADADOS")
print("="*60)

print("""
 SCHEMA DE METADADOS PARA CHUNKS

| Campo | Tipo | Descrição | Obrigatório |
|-------|------|-----------|-------------|
| fonte | string | Nome do arquivo .md de origem
| documento_id | string | Identificador único do documento
| chunk_index | integer | Posição do chunk no documento (0-based)
| estrategia | string | Estratégia de chunking usada
| chunk_size | integer | Tamanho configurado do chunk
| chunk_overlap | integer | Overlap configurado
| n_caracteres | integer | Tamanho real do chunk em caracteres
| secao | string | Seção/título do documento (se aplicável)
| nivel_header | integer | Nível do header (1-6) para markdown
| tipo_conteudo | string | Tipo: 'texto', 'codigo', 'tabela', 'lista'
| data_processamento | string | Data de criação do chunk

 CAMPOS PRÓPRIOS ADICIONADOS:

1. secao: Permite agrupar chunks por seção do documento.
   Pergunta que responde: "Quais chunks pertencem à seção X?"

2. nivel_header: Útil para chunks de markdown.
   Pergunta que responde: "Qual é a hierarquia estrutural deste chunk?"

3. tipo_conteudo: Classifica o tipo de conteúdo.
   Pergunta que responde: "Quais chunks são de código vs. texto puro?"

4. data_processamento: Rastreia quando o chunk foi criado.
   Pergunta que responde: "Este chunk foi gerado em qual versão do processamento?"
""")

# ============================================================
# 9. EXEMPLO DE CHUNK COM METADADOS
# ============================================================

print("\n" + "="*60)
print("EXEMPLO DE CHUNK REAL COM METADADOS")
print("="*60)

chunk_exemplo = {
    "chunk_id": "doc01_test02_chunk0015",
    "texto": "O transformer usa atenção multi-cabeça para processar sequências paralelamente.",
    "embedding": [0.0234, -0.0156, 0.0892, 0.0456, -0.0321],
    "metadata": {
        "fonte": "attention_is_all_you_need.md",
        "documento_id": "doc01",
        "chunk_index": 14,
        "estrategia": "fixed_500_no_overlap",
        "chunk_size": 500,
        "chunk_overlap": 0,
        "n_caracteres": 89,
        "secao": "3.2 - Multi-Head Attention",
        "nivel_header": 2,
        "tipo_conteudo": "texto",
        "data_processamento": "2026-08-12T21:30:00"
    }
}

print(json.dumps(chunk_exemplo, ensure_ascii=False, indent=2))

# ============================================================
# 10. RESPOSTAS EXERCÍCIO 2
# ============================================================

print("\n" + "="*60)
print("RESPOSTAS - EXERCÍCIO 2")
print("="*60)

print("\n Qual campo você incluiria se precisasse citar a fonte na resposta final do RAG?")
print(" Resposta: O campo 'fonte' é o principal para citação.")
print("   Usaria também 'documento_id', 'secao', e 'chunk_index' para citação completa:")
print("   'Fonte: attention_is_all_you_need.md, Seção 3.2, Chunk #15'")

print("\n Por que chunk_index é útil?")
print(" Resposta: O chunk_index é útil porque:")
print("   1. Permite reconstruir a ordem original do texto")
print("   2. Ajuda a identificar chunks adjacentes quando um chunk está cortado")
print("   3. Facilita a navegação entre chunks para obter contexto completo")
print("   4. Permite detectar se chunks consecutivos foram recuperados")
print("   5. Útil para debug e validação da estratégia de chunking")

# ============================================================
# 11. CONFIGURAÇÃO DOS EXPERIMENTOS
# ============================================================

EXPERIMENTOS = [
    {"test_id": 1, "estrategia": "fixed_200_no_overlap", "chunk_size": 200, "chunk_overlap": 0},
    {"test_id": 2, "estrategia": "fixed_500_no_overlap", "chunk_size": 500, "chunk_overlap": 0},
    {"test_id": 3, "estrategia": "fixed_1000_no_overlap", "chunk_size": 1000, "chunk_overlap": 0},
    {"test_id": 4, "estrategia": "fixed_2000_no_overlap", "chunk_size": 2000, "chunk_overlap": 0},
    {"test_id": 5, "estrategia": "fixed_500_overlap_50", "chunk_size": 500, "chunk_overlap": 50},
    {"test_id": 6, "estrategia": "fixed_500_overlap_200", "chunk_size": 500, "chunk_overlap": 200},
    {"test_id": 7, "estrategia": "paragraph", "chunk_size": 1000, "chunk_overlap": 0},
    {"test_id": 8, "estrategia": "sentences_3", "chunk_size": 500, "chunk_overlap": 0},
    {"test_id": 9, "estrategia": "recursive", "chunk_size": 500, "chunk_overlap": 0},
    {"test_id": 10, "estrategia": "markdown_headers", "chunk_size": 0, "chunk_overlap": 0},
]

# ============================================================
# 12. FUNÇÕES DE PROCESSAMENTO
# ============================================================

def clean_text(text: str) -> str:
    """Remove tokens especiais e caracteres de controle"""
    if not text:
        return ""
    tokens_especiais = ['<|endofprompt|>', '<|endoftext|>', '<|fim|>']
    for token in tokens_especiais:
        text = text.replace(token, '')
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_splitter(exp):
    """Retorna o splitter apropriado para cada experimento"""
    chunk_size = exp.get("chunk_size", 500)
    chunk_overlap = exp.get("chunk_overlap", 0)
    estrategia = exp.get("estrategia", "")

    if estrategia == "paragraph":
        separators = ["\n\n", "\n"]
    elif estrategia == "sentences_3":
        separators = [". ", "! ", "? ", "\n\n", "\n"]
    elif estrategia == "markdown_headers":
        return MarkdownHeaderTextSplitter(headers_to_split_on=[
            ("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")
        ])
    else:
        separators = ["\n\n", "\n", ".", "!", "?", ",", " ", ""]

    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators,
        length_function=len,
    )

def apply_splitter_safe(base_doc, exp):
    """Aplica o splitter de forma segura"""
    try:
        clean_content = clean_text(base_doc.page_content)
        clean_doc = Document(page_content=clean_content, metadata=base_doc.metadata)

        splitter = get_splitter(exp)

        if exp.get("estrategia") == "markdown_headers":
            try:
                return splitter.split_text(clean_content)
            except:
                return RecursiveCharacterTextSplitter(
                    chunk_size=500, chunk_overlap=0,
                    separators=["\n\n", "\n"], length_function=len
                ).split_documents([clean_doc])
        else:
            return splitter.split_documents([clean_doc])

    except Exception as e:
        print(f"    Erro no splitter: {e}. Usando fallback...")
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=exp.get("chunk_size", 500),
            chunk_overlap=exp.get("chunk_overlap", 0),
            separators=["\n\n", "\n", ".", " ", ""],
            length_function=len,
        )
        clean_content = clean_text(base_doc.page_content)
        clean_doc = Document(page_content=clean_content, metadata=base_doc.metadata)
        return splitter.split_documents([clean_doc])

def calculate_statistics(chunks):
    """Calcula estatísticas dos chunks"""
    if not chunks:
        return {"num_chunks": 0, "avg_chunk_size": 0, "min_chunk_size": 0, "max_chunk_size": 0}
    sizes = [len(c.page_content) if hasattr(c, 'page_content') else len(c) for c in chunks]
    import statistics
    return {
        "num_chunks": len(chunks),
        "avg_chunk_size": statistics.mean(sizes) if sizes else 0,
        "min_chunk_size": min(sizes) if sizes else 0,
        "max_chunk_size": max(sizes) if sizes else 0,
        "total_characters": sum(sizes) if sizes else 0
    }

def create_chunk_records(chunks, doc_id, doc_name, exp, secao=None):
    """Cria registros para cada chunk com metadados completos"""
    records = []
    for idx, chunk in enumerate(chunks):
        if hasattr(chunk, 'page_content'):
            text = chunk.page_content
            metadata = chunk.metadata.copy() if hasattr(chunk, 'metadata') else {}
        else:
            text = str(chunk)
            metadata = {}

        # Extrai seção do metadata se disponível
        secao_atual = secao or metadata.get("secao", metadata.get("Header 1", "indefinido"))

        # Adiciona metadados padronizados
        metadata.update({
            "fonte": f"{doc_name}.md",
            "documento_id": doc_id,
            "chunk_index": idx,
            "estrategia": exp.get("estrategia"),
            "chunk_size": exp.get("chunk_size"),
            "chunk_overlap": exp.get("chunk_overlap", 0),
            "n_caracteres": len(text),
            "secao": secao_atual,
            "data_processamento": datetime.now().isoformat()
        })

        records.append({
            "chunk_id": f"{doc_id}_test{exp['test_id']:02d}_chunk{idx:04d}",
            "page_content": text,
            "metadata": metadata
        })
    return records

# ============================================================
# 13. PROCESSAMENTO DE DOCUMENTOS
# ============================================================

def processar_documento(md_path: Path, doc_id: str, doc_name: str, output_base: Path):
    """Processa um documento com todas as estratégias"""

    print(f"\n Processando: {doc_name} ({md_path.name})")

    try:
        # Lê o texto
        text_content = md_path.read_text(encoding="utf-8", errors='ignore')
        text_content = clean_text(text_content)

        if not text_content:
            print(f"    Documento vazio! Pulando...")
            return None

        # Limita o tamanho para evitar processamento excessivo
        if len(text_content) > 100000:
            print(f"    Documento muito grande ({len(text_content)} caracteres). Truncando...")
            text_content = text_content[:100000]

        base_doc = Document(
            page_content=text_content,
            metadata={
                "documento_id": doc_id,
                "documento_nome": doc_name,
                "fonte": str(md_path),
            }
        )

        doc_output_dir = output_base / doc_id
        doc_output_dir.mkdir(parents=True, exist_ok=True)

        resultados = []
        total_chunks = 0

        for exp in EXPERIMENTOS:
            print(f"    Teste {exp['test_id']:02d} - {exp['estrategia']}")

            try:
                # Aplica splitter
                chunks = apply_splitter_safe(base_doc, exp)

                if not chunks:
                    print(f"       Nenhum chunk gerado!")
                    continue

                if isinstance(chunks[0], str):
                    chunks = [Document(page_content=c, metadata={}) for c in chunks]

                # Gera embeddings
                texts = [c.page_content for c in chunks]

                # Gera embeddings em lotes para evitar sobrecarga
                batch_size = 50
                all_embeddings = []
                for i in range(0, len(texts), batch_size):
                    batch = texts[i:i+batch_size]
                    batch_embeddings = embeddings.embed_documents(batch)
                    all_embeddings.extend(batch_embeddings)

                # Cria registros com metadados completos
                records = create_chunk_records(chunks, doc_id, doc_name, exp)

                # Adiciona embeddings aos registros
                for i, record in enumerate(records):
                    record["embedding"] = all_embeddings[i] if i < len(all_embeddings) else []

                # Salva o arquivo JSON
                test_dir = doc_output_dir / f"test_{exp['test_id']:02d}"
                test_dir.mkdir(parents=True, exist_ok=True)
                output_file = test_dir / "chunks_embeddings.json"
                output_file.write_text(
                    json.dumps(records, ensure_ascii=False, indent=2),
                    encoding="utf-8"
                )

                # Estatísticas
                stats = calculate_statistics(chunks)
                total_chunks += stats['num_chunks']
                print(f"       {stats['num_chunks']} chunks gerados")

                resultados.append({
                    "test_id": exp["test_id"],
                    "estrategia": exp["estrategia"],
                    **stats,
                })

            except Exception as e:
                print(f"       Erro: {str(e)[:100]}")
                continue

        # Summary do documento
        doc_summary = {
            "documento_id": doc_id,
            "documento": doc_name,
            "embedding_model": EMBEDDING_MODEL,
            "total_chunks": total_chunks,
            "experimentos": resultados,
            "processado_em": datetime.now().isoformat()
        }

        summary_path = doc_output_dir / "summary.json"
        summary_path.write_text(
            json.dumps(doc_summary, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        print(f"    Total: {total_chunks} chunks em {len(resultados)} experimentos")
        return doc_summary

    except Exception as e:
        print(f"    Erro ao processar: {e}")
        return None

# ============================================================
# 14. EXECUÇÃO PRINCIPAL
# ============================================================

print("\n" + "="*60)
print(" INICIANDO PROCESSAMENTO DOS DOCUMENTOS")
print("="*60)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

todos_resultados = []
documentos_processados = 0
documentos_com_erro = 0

for i, md_path in enumerate(md_files[:12], 1):  # Processa até 12 documentos
    doc_id = f"doc{i:02d}"
    doc_name = md_path.stem

    print(f"\n{'='*60}")
    print(f"[{i}/{min(len(md_files), 12)}] Processando: {doc_name}")
    print(f"{'='*60}")

    start_time = time.time()

    summary = processar_documento(md_path, doc_id, doc_name, RESULTS_DIR)

    elapsed = time.time() - start_time

    if summary:
        todos_resultados.append(summary)
        documentos_processados += 1
        print(f"   ⏱️  Tempo: {elapsed:.2f} segundos")
    else:
        documentos_com_erro += 1

# ============================================================
# 15. RELATÓRIO FINAL
# ============================================================

print("\n" + "="*60)
print(" RELATÓRIO FINAL")
print("="*60)

print(f"\n Documentos processados com sucesso: {documentos_processados}")
print(f" Documentos com erro: {documentos_com_erro}")

# Conta total de chunks
total_chunks_global = 0
for doc in todos_resultados:
    if doc and 'experimentos' in doc:
        for exp in doc['experimentos']:
            total_chunks_global += exp.get('num_chunks', 0)

print(f" Total de chunks gerados: {total_chunks_global}")
print(f" Experimentos por documento: {len(EXPERIMENTOS)}")

# Salva summary global
if todos_resultados:
    global_summary = {
        "projeto": "Avaliacao de Estrategias de Chunking",
        "embedding_model": EMBEDDING_MODEL,
        "num_documentos_processados": documentos_processados,
        "num_experimentos": len(EXPERIMENTOS),
        "total_chunks": total_chunks_global,
        "documentos": todos_resultados,
        "gerado_em": datetime.now().isoformat()
    }

    global_path = RESULTS_DIR / "summary.json"
    global_path.write_text(
        json.dumps(global_summary, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(f" Summary global salvo em: {global_path}")

    # Mostra detalhes dos documentos processados
    print("\n DOCUMENTOS PROCESSADOS:")
    for i, doc in enumerate(todos_resultados[:5], 1):
        print(f"   {i}. {doc.get('documento', 'Desconhecido')}")
        if 'experimentos' in doc and doc['experimentos']:
            total = sum(exp.get('num_chunks', 0) for exp in doc['experimentos'])
            print(f"      └─ {len(doc['experimentos'])} experimentos, {total} chunks")

    if len(todos_resultados) > 5:
        print(f"   ... e mais {len(todos_resultados)-5} documentos")
else:
    print(" Nenhum documento foi processado com sucesso!")

print("\n" + "="*60)
print(" PROCESSAMENTO CONCLUÍDO!")
print("="*60)

# ============================================================
# 16. VERIFICAÇÃO DOS ARQUIVOS GERADOS
# ============================================================

print("\n VERIFICANDO ARQUIVOS GERADOS:")

for doc_dir in sorted(RESULTS_DIR.glob("doc*")):
    print(f"\n {doc_dir.name}/")

    # Summary do documento
    summary_file = doc_dir / "summary.json"
    if summary_file.exists():
        data = json.loads(summary_file.read_text())
        print(f"   └─ summary.json: {data.get('total_chunks', 0)} chunks")

    # Lista os testes
    test_dirs = sorted(doc_dir.glob("test_*"))
    for test_dir in test_dirs:
        json_file = test_dir / "chunks_embeddings.json"
        if json_file.exists():
            data = json.loads(json_file.read_text())
            print(f"   └─ {test_dir.name}/: {len(data)} chunks")

print("\n" + "="*60)
print(" TAREFA CONCLUÍDA COM SUCESSO!")
print("="*60)

 Dependências instaladas!
 Bibliotecas importadas!

 DESCOMPACTANDO ARQUIVOS
 Arquivos descompactados em: /content/documentos_extraidos

 PROCURANDO ARQUIVOS MARKDOWN
 Encontrados 0 arquivos para processar

 CARREGANDO MODELO DE EMBEDDINGS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Modelo carregado: sentence-transformers/all-MiniLM-L6-v2
 Dimensão: 384

EXERCÍCIO 1 - CRIANDO DOCUMENTS NA MÃO

 Documentos manuais criados: 6

 LISTA DE DOCUMENTOS:

Documento 1:
  Conteúdo: Embeddings são representações vetoriais densas de texto que capturam significado...
  Metadados: {'fonte': 'arquivo_01.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'embeddings', 'autor': 'Aluno', 'dificuldade': 'intermediario', 'tags': ['embedding', 'vetor', 'semantica']}

Documento 2:
  Conteúdo: O chunking é o processo de dividir documentos grandes em pedaços menores e geren...
  Metadados: {'fonte': 'arquivo_01.md', 'pagina': 2, 'tipo': 'pratica', 'tema': 'chunking', 'autor': 'Aluno', 'dificuldade': 'iniciante', 'tags': ['chunking', 'divisao', 'documentos']}

Documento 3:
  Conteúdo: RAG (Retrieval-Augmented Generation) combina busca semântica com geração de text...
  Metadados: {'fonte': 'arquivo_02.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'rag', 'autor': 'Aluno', 'dificuldade': 'avancado'